# Notebook 4 – Duplicate Data Handling

## 1. What Are Duplicate Records?

Duplicate records are rows in a dataset that repeat information already present in other rows.

Duplicates may occur because of repeated data entry, system errors, data integration, repeated transactions, or multiple records being collected from the same source.

Duplicate data can affect statistical analysis, reporting, and Machine Learning if the repeated records do not represent independent observations.

## 2. Types of Duplicates

### Exact Duplicates

Exact duplicates are rows in which every column contains the same value.

Example:

| CustomerID | Age | Contract |
| ---------- | --: | -------- |
| 101        |  25 | Monthly  |
| 102        |  30 | Yearly   |
| 102        |  30 | Yearly   |

The last two rows are exact duplicates.

### Partial Duplicates

Partial duplicates are records that match on selected columns but differ in other columns.

Example:

| CustomerID | Name | City       |
| ---------- | ---- | ---------- |
| 101        | Arun | Chennai    |
| 101        | Arun | Coimbatore |

These records share a CustomerID and name but have different cities.

They may represent an incorrect duplicate, an updated address, or legitimate information. Further investigation is necessary.

## 3. Why Do Duplicates Occur?

Duplicates may result from:

* Repeated manual data entry.
* Repeated form submissions.
* Data being imported multiple times.
* Combining multiple databases.
* System synchronization problems.
* Repeated transactions.
* Incorrect record-matching rules.

Understanding the cause helps determine whether a record should be removed, merged, retained, or investigated.

## 4. Identifying Duplicates Using Pandas

### duplicated()

The `duplicated()` method identifies duplicate rows.

Example:

`df.duplicated()`

It returns Boolean values indicating which rows are considered duplicates. By default, the first occurrence is marked `False`, and subsequent matching rows are marked `True`.

### Counting Duplicates

`df.duplicated().sum()`

This counts rows marked as duplicates under the default rules.

### Checking Duplicates Based on Selected Columns

`df.duplicated(subset=["CustomerID"])`

This identifies repeated CustomerID values, even if other columns differ.

The selected columns should represent an appropriate business key or matching rule.

## 5. Handling Duplicate Records

### drop_duplicates()

The `drop_duplicates()` method removes duplicate rows according to the selected columns.

Example:

`df.drop_duplicates()`

By default, it retains the first occurrence of an exact duplicate and removes subsequent occurrences.

### Keeping the Last Record

`df.drop_duplicates(keep="last")`

This retains the last occurrence among matching rows.

### Removing All Matching Records

`df.drop_duplicates(keep=False)`

This removes every row belonging to a duplicated group.

### Duplicate Detection Using Selected Columns

`df.drop_duplicates(subset=["CustomerID"])`

This keeps one row per CustomerID according to the selected retention rule.

However, this operation should only be used when CustomerID is expected to identify one unique record.

## 6. Business Rules for Duplicate Removal

Duplicate removal should follow business rules rather than relying only on matching values.

Examples:

* A customer master table may require one record per CustomerID.
* An order table may legitimately contain multiple orders from the same customer.
* A transaction table may contain repeated amounts that represent separate valid transactions.
* A customer table may contain updated records that should be merged rather than deleted.

Before removing duplicates, determine what makes a record unique.

## 7. Why Blindly Deleting Duplicates Is Dangerous

Blind duplicate removal can cause problems.

### Loss of Valid Information

Two records may look similar but represent separate real-world events.

### Loss of Updated Information

A later record may contain more recent or more accurate information.

### Incorrect Aggregations

Removing valid repeated transactions can produce incorrect revenue, purchase-count, or activity calculations.

### Incorrect Machine Learning Results

Duplicate observations can influence model training and evaluation. However, whether this is harmful depends on what the records represent and how the dataset is split.

For example, repeated or near-identical records appearing in both training and test data can produce overly optimistic evaluation results.

## 8. Recommended Duplicate Handling Workflow

1. Inspect the dataset.
2. Identify exact duplicates.
3. Identify possible partial duplicates using suitable keys.
4. Investigate the meaning of repeated records.
5. Apply appropriate business rules.
6. Remove or merge confirmed accidental duplicates.
7. Validate the result.
8. Document the decision and its impact.

## Conclusion

Duplicate data handling is an important preprocessing step, but not every repeated value represents an error.

The goal is to identify and resolve unintended duplicate records while preserving legitimate observations and important business information.


# Demonstration Duplicate Data Handling

## Simple Example

Consider the following customer dataset.

| CustomerID | Name  | City       | MonthlyCharges |
| ---------- | ----- | ---------- | -------------: |
| 101        | Arun  | Chennai    |            500 |
| 102        | Priya | Coimbatore |            700 |
| 102        | Priya | Coimbatore |            700 |
| 103        | Kumar | Salem      |            600 |
| 103        | Kumar | Erode      |            600 |

### Problems Identified

1. Customer 102 has an exact duplicate record.
2. Customer 103 appears twice with different city values.
3. Repeated CustomerID values require investigation.

### Analysis

The repeated record for Customer 102 appears identical across every column. If it is an accidental duplicate, one copy can be removed.

Customer 103 has the same ID, name, and monthly charges but different cities. These records may indicate an address change, a data-entry error, or multiple legitimate records.

The correct decision depends on the business rules and the meaning of CustomerID.

### Technique Selected

* Use `duplicated()` to identify exact duplicate records.
* Use `duplicated(subset=["CustomerID"])` to identify repeated customer IDs.
* Use `drop_duplicates()` to remove confirmed exact duplicates.
* Investigate conflicting records before deciding how to resolve them.

## Real-World Business Example

A telecommunications company combines customer records from different systems.

The same customer may appear more than once because the records were imported repeatedly.

If accidental duplicates remain, the company may overestimate its customer count or give repeated records excessive influence during analysis.

However, multiple billing transactions from the same customer may be valid and must not be deleted simply because the customer ID repeats.

## AI/ML Use Case

For customer churn prediction:

* Confirmed accidental duplicate customer records may need to be removed or reconciled.
* Conflicting customer records should be investigated.
* Repeated transactions should be retained when they represent separate events.
* Similar records should be checked when splitting training and test data to avoid overly optimistic evaluation caused by duplicated observations.

**Important:** Duplicate removal should be based on the meaning of the data, not just on the presence of repeated values.


In [1]:
import pandas as pd

# Create a sample customer dataset
data = {
    "CustomerID": [101, 102, 102, 103, 103, 104],
    "Name": ["Arun", "Priya", "Priya", "Kumar", "Kumar", "Meena"],
    "City": [
        "Chennai",
        "Coimbatore",
        "Coimbatore",
        "Salem",
        "Erode",
        "Madurai"
    ],
    "MonthlyCharges": [500, 700, 700, 600, 600, 800]
}

df = pd.DataFrame(data)

# Step 1: Display the original dataset
print("ORIGINAL DATASET")
print(df)

# Step 2: Inspect dataset dimensions
print("\nOriginal Shape:", df.shape)

# Step 3: Identify exact duplicate records
print("\nEXACT DUPLICATE MASK")
print(df.duplicated())

# Step 4: Count exact duplicate records
print("\nNumber of Exact Duplicates:", df.duplicated().sum())

# Step 5: Display exact duplicate rows
print("\nEXACT DUPLICATE ROWS")
print(df[df.duplicated(keep=False)])

# Step 6: Identify repeated CustomerID values
print("\nRECORDS WITH REPEATED CUSTOMER IDs")
print(
    df[df.duplicated(subset=["CustomerID"], keep=False)]
    .sort_values("CustomerID")
)

# Step 7: Remove confirmed exact duplicate records
clean_df = df.drop_duplicates().copy()

print("\nDATASET AFTER REMOVING EXACT DUPLICATES")
print(clean_df)

# Step 8: Compare original and cleaned datasets
print("\nBEFORE CLEANING")
print("Shape:", df.shape)
print("Exact duplicates:", df.duplicated().sum())

print("\nAFTER CLEANING")
print("Shape:", clean_df.shape)
print("Exact duplicates:", clean_df.duplicated().sum())

# Step 9: Demonstrate keeping the last record per CustomerID
# This is only an example of the method.
# Use it only when business rules justify keeping the last record.
last_record_df = df.drop_duplicates(
    subset=["CustomerID"],
    keep="last"
)

print("\nKEEPING THE LAST RECORD PER CUSTOMER ID")
print(last_record_df)

# Step 10: Demonstrate removing all rows belonging to
# groups with repeated CustomerID values
unique_id_records = df.drop_duplicates(
    subset=["CustomerID"],
    keep=False
)

print("\nCUSTOMER IDs THAT APPEAR ONLY ONCE")
print(unique_id_records)

ORIGINAL DATASET
   CustomerID   Name        City  MonthlyCharges
0         101   Arun     Chennai             500
1         102  Priya  Coimbatore             700
2         102  Priya  Coimbatore             700
3         103  Kumar       Salem             600
4         103  Kumar       Erode             600
5         104  Meena     Madurai             800

Original Shape: (6, 4)

EXACT DUPLICATE MASK
0    False
1    False
2     True
3    False
4    False
5    False
dtype: bool

Number of Exact Duplicates: 1

EXACT DUPLICATE ROWS
   CustomerID   Name        City  MonthlyCharges
1         102  Priya  Coimbatore             700
2         102  Priya  Coimbatore             700

RECORDS WITH REPEATED CUSTOMER IDs
   CustomerID   Name        City  MonthlyCharges
1         102  Priya  Coimbatore             700
2         102  Priya  Coimbatore             700
3         103  Kumar       Salem             600
4         103  Kumar       Erode             600

DATASET AFTER REMOVING EXACT DUPLI